<sub>🐍 **Python for Django** · by **Rifat Islam Sourov** · Chapter 09</sub>

# 🎁 Decorators

## 🎯 What you'll learn
- Functions can be passed around like any other value
- What a **decorator** is and how `@` works
- How Django's `@login_required` works behind the scenes

---
## 1. Functions are values too 🤯
You can put a function in a variable or pass it to another function, just like a number or a string.

In [ ]:
def shout(text):
    return text.upper() + "!"

yell = shout             # no brackets = we're not calling it, just passing it
print(yell("hello"))

In [ ]:
def apply_twice(func, value):
    return func(func(value))

def add_five(x):
    return x + 5

print(apply_twice(add_five, 10))   # 20

## 2. Functions inside functions 🪆

In [ ]:
def outer():
    def inner():
        return "I'm inside! 🏠"
    return inner          # give back the inner function

my_func = outer()
print(my_func())

## 3. What is a decorator? 🎁
A decorator is like **gift wrapping**. You take a function, wrap it in extra paper (extra code that runs before or after), and hand it back. The present inside is the same, it just has something extra around it.

In [ ]:
def wrap_with_stars(func):
    def wrapper():
        print("⭐⭐⭐⭐⭐")
        func()                     # run the original function
        print("⭐⭐⭐⭐⭐")
    return wrapper

def say_hi():
    print("   Hi there!")

say_hi = wrap_with_stars(say_hi)   # wrap it
say_hi()

## 4. The `@` shortcut ✨
Writing `say_hi = wrap_with_stars(say_hi)` is a bit clumsy. Python gives us a shortcut: put **`@decorator_name`** above the function.

In [ ]:
@wrap_with_stars
def celebrate():
    print("   I learned decorators! 🎉")

celebrate()

`@wrap_with_stars` above `def celebrate` means exactly the same as `celebrate = wrap_with_stars(celebrate)`.

## 5. Decorators for functions with arguments
Use `*args, **kwargs` (Chapter 05!) so the wrapper accepts **any** arguments. Also use `functools.wraps` to keep the original function's name.

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"⏱️ {func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def add_up(n):
    return sum(range(n))

print(add_up(1_000_000))

## 6. 🔐 Let's build our own `login_required`!
This is a simplified version of what Django does:

In [ ]:
from functools import wraps

def login_required(view_func):
    @wraps(view_func)
    def wrapper(user, *args, **kwargs):
        if not user.get("is_logged_in"):
            return "🔒 Please log in first!"
        return view_func(user, *args, **kwargs)
    return wrapper

@login_required
def dashboard(user):
    return f"📊 Welcome to your dashboard, {user['name']}!"

guest = {"name": "Guest", "is_logged_in": False}
rifat = {"name": "Rifat", "is_logged_in": True}

print(dashboard(guest))
print(dashboard(rifat))

The `dashboard` function only cares about showing the dashboard. The **decorator** handles the security check. Clean! 🧼

> ### 🔗 Django Connection
>
> You'll put decorators on top of your views:
>
> ```python
> from django.contrib.auth.decorators import login_required
> from django.views.decorators.http import require_POST
>
> @login_required                  # only logged-in users can see this page
> def profile(request):
>     return render(request, "profile.html")
>
> @require_POST                    # only allow form submissions
> def delete_post(request, pk):
>     ...
> ```
>
> And in models you'll see `@property`, a built-in decorator that lets a method be used like an attribute:
> ```python
> class Person(models.Model):
>     first_name = models.CharField(max_length=50)
>     last_name = models.CharField(max_length=50)
>
>     @property
>     def full_name(self):
>         return f"{self.first_name} {self.last_name}"
> # person.full_name   ← no brackets needed!
> ```

---
## 🏋️ Practice Time!

**Exercise 1:** Write a decorator `say_before_after` that prints `"Before ⏪"` before the function runs and `"After ⏩"` after. Use it on a function that prints `"Running!"`.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
def say_before_after(func):
    def wrapper(*args, **kwargs):
        print("Before ⏪")
        result = func(*args, **kwargs)
        print("After ⏩")
        return result
    return wrapper

@say_before_after
def run():
    print("Running!")

run()
```

</details>

**Exercise 2:** Write a decorator `admin_only` for functions that take a `user` dictionary. If `user["role"]` isn't `"admin"`, return `"⛔ Admins only!"`.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
from functools import wraps

def admin_only(func):
    @wraps(func)
    def wrapper(user, *args, **kwargs):
        if user.get("role") != "admin":
            return "⛔ Admins only!"
        return func(user, *args, **kwargs)
    return wrapper

@admin_only
def delete_everything(user):
    return f"🗑️ {user['name']} deleted everything!"

print(delete_everything({"name": "Sara", "role": "student"}))
print(delete_everything({"name": "Rifat", "role": "admin"}))
```

</details>

**Exercise 3:** Create a class `Circle` with a `radius`, and an `area` **property** using `@property`, so that `Circle(2).area` works without brackets.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14 * self.radius ** 2

print(Circle(2).area)   # 12.56
```

</details>

---
## ✅ Recap
- Functions are values: you can pass them around and return them.
- A **decorator** wraps a function to add extra behaviour.
- `@my_decorator` above a function = `func = my_decorator(func)`.
- Use `*args, **kwargs` and `@wraps` inside decorators.
- Django: `@login_required`, `@require_POST`, `@property`.

---
### ⏭️ Next up: [📂 Files and JSON](../10-Files-JSON-and-Dates/01-Files-and-JSON.ipynb)

<sub>Made with ❤️ by Rifat Islam Sourov · [Back to the course home](../README.md)</sub>